In [ ]:
#@title << Run this cell first — environment setup {display-mode: "form"}
import sys, os

if "google.colab" in sys.modules:
    !git clone --quiet --single-branch --branch main https://github.com/YanickSchraner/CAS-DeepRL.git
    !cp -r "CAS-DeepRL/Tag 2/envs" .
    sys.path.insert(0, ".")
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

# Unit: Advantage Actor-Critic (A2C) — Smart Building HVAC Control 🏢🌡️

## The Real-World Problem

Buildings account for **~40% of global energy consumption**, and heating/cooling is the largest single component. Smart thermostats and HVAC controllers are one of the most commercially viable RL applications today.

In **2016, Google DeepMind applied RL to the cooling systems of Google's data centers** and achieved a **40% reduction in cooling energy consumption**. The agent learned to pre-cool when electricity is cheap and reduce cooling during peak-price hours — something rule-based systems couldn't do efficiently.

**Your task:** Train an A2C agent to control a room heater. The agent must:
- Keep the room temperature close to **21°C** (comfort target)
- Minimize **electricity cost** (prices vary throughout the day — cheap at night, expensive in the evening)
- These two objectives are in tension: heating is cheapest when you least need it

### 📚 RL Library: [Stable-Baselines3](https://stable-baselines3.readthedocs.io/)

## Objectives 🏆

By the end of this notebook you will:
- Understand the A2C algorithm and when to use it
- Train an RL agent on a **custom gymnasium environment** with domain-relevant dynamics
- **Investigate the effect of hyperparameters** on training
- **Redesign the reward function** to change agent behaviour
- Diagnose a **poorly-trained agent** and explain why it failed

## Install and import

In [ ]:
!pip install stable-baselines3[extra] gymnasium matplotlib --quiet

In [ ]:
import sys, os

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium.utils.env_checker import check_env

from stable_baselines3 import A2C
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback

from envs.hvac_env import HvacEnv

## Part 1 — Explore the Environment

Before training, always **understand your environment**. Run the cell below and observe the observation space, action space, and what a random agent does.

In [ ]:
env = HvacEnv()
check_env(env)  # validates the environment follows the gymnasium API

print("=== HVAC Environment ===")
print(f"Observation space : {env.observation_space}")
print(f"  [room_temp, outdoor_temp, hour_sin, hour_cos, price_norm]")
print(f"Action space      : {env.action_space}  (0=OFF, 1=LOW 0.5kW, 2=MED 1.5kW, 3=HIGH 3.0kW)")
print(f"Episode length    : {env.episode_length} steps (= 1 day in 15-min intervals)")
print()

# Sample one observation
obs, _ = env.reset(seed=42)
print("Sample observation:", obs)
print("  room_temp    :", obs[0], "°C")
print("  outdoor_temp :", obs[1], "°C")
print("  hour_sin/cos :", obs[2], obs[3])
print("  price_norm   :", obs[4], "(0=cheapest, 1=most expensive)")

In [ ]:
# Run a random agent for one episode and track temperature + reward
env = HvacEnv(seed=42)
obs, _ = env.reset()

temps, actions_taken, rewards_list, hours = [], [], [], []
done = False
hour = 0.0

while not done:
    action = env.action_space.sample()  # random action
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    temps.append(info["room_temp"])
    actions_taken.append(info["heater_kw"])
    rewards_list.append(reward)
    hour += 0.25
    hours.append(hour)

print(f"Random agent total reward: {sum(rewards_list):.1f}")
print(f"Average room temp: {np.mean(temps):.1f}°C  (target: 21°C)")
print(f"Time outside comfort zone (|T-21|>1): {np.mean(np.abs(np.array(temps)-21)>1)*100:.0f}% of steps")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(hours, temps, label="Room temp")
ax1.axhline(21, color="green", linestyle="--", label="Target 21°C")
ax1.fill_between(hours, 20, 22, alpha=0.1, color="green", label="Comfort zone")
ax1.set_ylabel("Temperature (°C)")
ax1.legend()
ax1.set_title("Random Agent — Room Temperature")

ax2.bar(hours, actions_taken, width=0.2, label="Heater power (kW)")
ax2.set_ylabel("Heater power (kW)")
ax2.set_xlabel("Hour of day")
ax2.legend()

plt.tight_layout()
plt.show()

**Observation:** The random agent switches the heater on and off erratically. Temperature fluctuates widely. This is your baseline — the agent you need to beat.

❓ **Question before training:** What strategy would *you* use if you were the controller? When would you heat more? When less?

## Part 2 — A2C: The Algorithm

**Advantage Actor-Critic (A2C)** combines two networks:
- **Actor**: outputs a probability distribution over actions (the *policy*)
- **Critic**: estimates the value function V(s) — how good is this state?

The **Advantage** A(s,a) = Q(s,a) - V(s) answers: *"Is this action better or worse than average in this state?"*
- A > 0 → this action led to better-than-expected return → increase its probability
- A < 0 → worse than expected → decrease its probability

**Why A2C over REINFORCE?**  
REINFORCE uses the full discounted return G_t as the signal, which is very noisy (high variance). The critic's value estimate V(s) acts as a **baseline** that reduces variance while keeping the gradient unbiased.

**Why A2C over Q-Learning/DQN?**  
A2C works natively with continuous and high-dimensional action spaces. DQN requires a discrete action space and can't scale as easily to complex policies.

## Part 3 — Train with Stable-Baselines3

In [ ]:
# Vectorised training environment (4 parallel environments = faster data collection)
train_env = make_vec_env(HvacEnv, n_envs=4)

# Separate eval environment (important: never evaluate on the training env)
eval_env = Monitor(HvacEnv(seed=99))

# Evaluation callback: saves the best model automatically
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path="./logs/hvac_a2c/",
    eval_freq=2000,
    n_eval_episodes=5,
    verbose=0,
)

# Create and train the agent
model = A2C(
    policy="MlpPolicy",
    env=train_env,
    learning_rate=7e-4,
    n_steps=5,           # steps to collect before each update
    gamma=0.99,          # discount factor
    ent_coef=0.01,       # entropy bonus for exploration
    verbose=1,
    seed=42,
)

model.learn(total_timesteps=200_000, callback=eval_callback)
print("Training complete.")

In [ ]:
# Evaluate the trained agent
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=20)
print(f"Trained A2C agent: {mean_reward:.1f} ± {std_reward:.1f} per episode")

# Compare with random baseline
def random_agent_reward(n_episodes=20, seed=0):
    rng = np.random.default_rng(seed)
    rewards = []
    for ep in range(n_episodes):
        env = HvacEnv(seed=ep)
        obs, _ = env.reset()
        total, done = 0, False
        while not done:
            obs, r, term, trunc, _ = env.step(env.action_space.sample())
            total += r
            done = term or trunc
        rewards.append(total)
    return np.mean(rewards), np.std(rewards)

rand_mean, rand_std = random_agent_reward()
print(f"Random agent     : {rand_mean:.1f} ± {rand_std:.1f} per episode")
improvement = (mean_reward - rand_mean) / abs(rand_mean) * 100
print(f"Improvement      : {improvement:.1f} %")

In [ ]:
# Visualise the trained agent's behaviour over one day
env_vis = HvacEnv(seed=7)
obs, _ = env_vis.reset()

temps, powers, prices, hours = [], [], [], []
done = False
hour = 0.0

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, info = env_vis.step(action)
    done = terminated or truncated
    temps.append(info["room_temp"])
    powers.append(info["heater_kw"])
    prices.append(info["price_eur_kwh"])
    hour += 0.25
    hours.append(hour)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(hours, temps, color="tomato")
axes[0].axhline(21, color="green", linestyle="--", label="Target 21°C")
axes[0].fill_between(hours, 20, 22, alpha=0.15, color="green")
axes[0].set_ylabel("Temperature (°C)")
axes[0].set_title("Trained A2C Agent — One Day")
axes[0].legend()

axes[1].bar(hours, powers, width=0.2, color="steelblue")
axes[1].set_ylabel("Heater power (kW)")

axes[2].plot(hours, prices, color="orange")
axes[2].set_ylabel("Electricity price (€/kWh)")
axes[2].set_xlabel("Hour of day")

plt.tight_layout()
plt.show()

❓ **Discuss:** Does the trained agent pre-heat when electricity is cheap? How does it respond to the price peak in the afternoon? Compare this to what you'd expect from a naive thermostat.

## Part 4 — Hyperparameter Investigation

One of the most important skills of an RL engineer is understanding **what each hyperparameter does** and how to tune it.

### Task: Investigate `ent_coef` (entropy coefficient)

The entropy coefficient controls how much the agent is encouraged to **explore** vs. **exploit**.
- High `ent_coef` → agent keeps trying different actions (more exploration)
- Low `ent_coef` → agent commits to its best-known action (more exploitation)

Train three agents with different `ent_coef` values and compare their final performance.

In [ ]:
ent_coef_values = [0.0, 0.01, 0.1]  # feel free to add more
results = {}

for ent_coef in ent_coef_values:
    print(f"Training with ent_coef={ent_coef}...")
    train_env = make_vec_env(HvacEnv, n_envs=4)
    eval_env_hp = Monitor(HvacEnv(seed=99))
    
    m = A2C("MlpPolicy", train_env, learning_rate=7e-4, ent_coef=ent_coef, verbose=0, seed=42)
    m.learn(total_timesteps=100_000)
    
    mean, std = evaluate_policy(m, eval_env_hp, n_eval_episodes=10)
    results[ent_coef] = (mean, std)
    print(f"  ent_coef={ent_coef}: mean reward = {mean:.1f} ± {std:.1f}")

print("\n--- Summary ---")
for k, (m, s) in sorted(results.items()):
    print(f"ent_coef={k:4}: {m:.1f} ± {s:.1f}")

❓ **Reflect:**
1. Which `ent_coef` performed best?
2. What happens with `ent_coef=0`? Can the agent still learn?
3. What's the risk of very high entropy (e.g., `ent_coef=1.0`)?

*Bonus:* Also try varying `learning_rate` (e.g., 1e-3, 7e-4, 1e-4) and observe the effect.

## Part 5 — Reward Shaping Challenge 🎯

The current reward is:
```
reward = -(discomfort_penalty + electricity_cost)
discomfort_penalty = |room_temp - 21°C| × 2.0
```

**Your task:** Modify the `HvacEnv` reward to create an agent with a *different* priority. Choose one:

**Option A — Cost-first agent:** Increase the electricity cost weight so the agent strongly prefers heating during off-peak hours even at the cost of some discomfort.

**Option B — Comfort-first agent:** Make discomfort extremely expensive so the agent keeps temperature within ±0.5°C of 21°C at all times, regardless of price.

**Option C — Night-mode agent:** Add a time-based weight that makes comfort less important between 23:00–06:00 (people are sleeping, ±2°C is acceptable).

**How to modify the reward:** Create a subclass of `HvacEnv` and override the `step` method, or copy the env file and adjust the reward formula.

In [ ]:
from envs.hvac_env import HvacEnv

class HvacEnvCustomReward(HvacEnv):
    """Your custom reward variant."""

    def step(self, action: int):
        obs, reward, terminated, truncated, info = super().step(action)

        # TODO: Replace the reward with your custom version.
        # Useful values from info:
        #   info["room_temp"]     — current room temperature after heater
        #   info["price_eur_kwh"] — current electricity price
        #   info["heater_kw"]     — power used this step
        #   self._hour            — current hour (0.0–24.0)

        custom_reward = reward  # <-- replace this

        return obs, custom_reward, terminated, truncated, info


# Train your custom agent
train_env_custom = make_vec_env(HvacEnvCustomReward, n_envs=4)
eval_env_custom = Monitor(HvacEnvCustomReward(seed=99))

model_custom = A2C("MlpPolicy", train_env_custom, learning_rate=7e-4, ent_coef=0.01, verbose=0, seed=42)
model_custom.learn(total_timesteps=150_000)

mean_custom, std_custom = evaluate_policy(model_custom, eval_env_custom, n_eval_episodes=20)
print(f"Custom reward agent: {mean_custom:.1f} ± {std_custom:.1f}")

❓ **Discuss:** How did changing the reward function change the agent's behaviour? Compare the temperature curves. In a real building, who gets to decide what the reward function should be — the ML engineer or the building manager?

## Part 6 — What Went Wrong? 🔍

Below is a poorly-trained agent. Run the evaluation and observe its behaviour, then answer the diagnostic questions.

In [ ]:
# A deliberately bad training run — high learning rate, no entropy
train_env_bad = make_vec_env(HvacEnv, n_envs=4)
bad_model = A2C(
    "MlpPolicy",
    train_env_bad,
    learning_rate=0.1,   # way too high
    ent_coef=0.0,        # no exploration
    verbose=0,
    seed=42,
)
bad_model.learn(total_timesteps=50_000)

# Evaluate bad agent
eval_env_bad = Monitor(HvacEnv(seed=5))
mean_bad, std_bad = evaluate_policy(bad_model, eval_env_bad, n_eval_episodes=20)
print(f"Bad agent reward : {mean_bad:.1f} ± {std_bad:.1f}")
print(f"Good agent reward: {mean_reward:.1f} ± {std_reward:.1f}")

# Visualise bad agent's behaviour
env_bad_vis = HvacEnv(seed=7)
obs, _ = env_bad_vis.reset()
temps_bad, powers_bad, hours_bad = [], [], []
done = False; hour = 0.0
while not done:
    action, _ = bad_model.predict(obs, deterministic=True)
    obs, _, term, trunc, info = env_bad_vis.step(action)
    done = term or trunc
    temps_bad.append(info["room_temp"])
    powers_bad.append(info["heater_kw"])
    hour += 0.25; hours_bad.append(hour)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
ax1.plot(hours_bad, temps_bad, color="red", label="Bad agent")
ax1.plot(hours, temps, color="green", alpha=0.6, label="Good agent")
ax1.axhline(21, color="black", linestyle="--", label="Target 21°C")
ax1.set_ylabel("Temperature (°C)")
ax1.legend()
ax1.set_title("Bad vs. Good Agent")

ax2.bar(hours_bad, powers_bad, width=0.2, color="red", alpha=0.7, label="Bad agent")
ax2.set_ylabel("Heater power (kW)")
ax2.set_xlabel("Hour of day")
ax2.legend()
plt.tight_layout()
plt.show()

❓ **Diagnostic questions:**
1. What action does the bad agent take most of the time? Can you see it in the chart?
2. Which of the two hyperparameter problems (`learning_rate=0.1` or `ent_coef=0.0`) do you think is more damaging? Why?
3. What would you change first to fix it? Try it!

**Tip:** An agent that always picks the same action may have:
- Converged too early to a suboptimal policy (no exploration)
- Gradient exploded due to too-high learning rate, causing random weights

These look similar from the outside but have different fixes.

## Summary

| What you did | RL Engineer skill |
|---|---|
| Explored HvacEnv before training | Always understand your environment first |
| Trained A2C with SB3 in 5 lines | Use production tools instead of reinventing |
| Compared trained vs. random agent | Always have a baseline |
| Swept `ent_coef` values | Hyperparameter sensitivity analysis |
| Customised the reward function | Reward engineering — the hardest real-world RL skill |
| Diagnosed the bad agent | Debugging is 50% of the job |

**Next:** In the PPO notebook, you'll apply similar skills to a **dynamic pricing** problem.